# Assignment 3 — Two Averages, One Argument

This notebook reproduces the Week 3 calculations from the cleaned Week 1 and Week 2 files.

**Important:** the dataset is illustrative teaching data, not actual PBS observations.


In [ ]:
import pandas as pd
from IPython.display import display

week1 = pd.read_csv("teaching_prices_clean_week1.csv", parse_dates=["date"])
week2 = pd.read_csv("teaching_prices_week2_clean.csv", parse_dates=["date"])

print("Week 1 rows and columns:", week1.shape)
print("Week 2 rows and columns:", week2.shape)
print("\\nWeek 1 missing values:")
display(week1.isna().sum().to_frame("missing"))
print("\\nWeek 2 missing values:")
display(week2.isna().sum().to_frame("missing"))


## 1. Week 1: mean, median and first-to-last percentage change

Mean and median are calculated from all available prices for each item across both cities. For percentage change, I first calculate the average of the two city prices on each date, then compare the first and last dates.

Formula: `((last-date average - first-date average) / first-date average) * 100`.


In [ ]:
results = []

for item, group in week1.groupby("item"):
    values = group["price"].dropna()
    date_average = group.groupby("date")["price"].mean().sort_index()

    first_date = date_average.index[0]
    last_date = date_average.index[-1]
    first_average = date_average.iloc[0]
    last_average = date_average.iloc[-1]

    results.append({
        "item": item,
        "valid_observations": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "first_date": first_date.date(),
        "first_date_average": first_average,
        "last_date": last_date.date(),
        "last_date_average": last_average,
        "percentage_change": (last_average-first_average)/first_average*100
    })

week1_summary = pd.DataFrame(results).sort_values("item")
display(week1_summary.round(2))


## 2. Week 2: Staple A with and without the flagged value

The Week 2 cleaned file contains a value of 1,250 for Staple A. The first calculation keeps it. The second calculation leaves out only that value so its effect can be compared. This is a sensitivity check, not a claim that the value is definitely incorrect.


In [ ]:
staple_a = week2.loc[week2["item"].eq("Staple A"), "price"].dropna()
staple_a_without_flag = staple_a.loc[staple_a.ne(1250)]

week2_comparison = pd.DataFrame([
    {
        "version": "With flagged value 1,250",
        "n": len(staple_a),
        "mean": staple_a.mean(),
        "median": staple_a.median()
    },
    {
        "version": "Without flagged value 1,250",
        "n": len(staple_a_without_flag),
        "mean": staple_a_without_flag.mean(),
        "median": staple_a_without_flag.median()
    }
])
display(week2_comparison.round(2))

print("Flagged row:")
display(week2.loc[
    week2["item"].eq("Staple A") & week2["price"].eq(1250),
    ["date", "item", "city", "price", "unit"]
])


## 3. Check the effect of the flagged value

The mean uses the magnitude of every observation, so a very high value can pull it upward. The median is based on the middle-ranked value(s), so it is usually less affected by one extreme observation.


In [ ]:
print("Mean difference:", round(staple_a.mean()-staple_a_without_flag.mean(), 2))
print("Median difference:", round(staple_a.median()-staple_a_without_flag.median(), 2))

assert len(staple_a) == 24
assert len(staple_a_without_flag) == 23
assert staple_a.eq(1250).sum() == 1
print("Checks passed.")


## 4. Interpretation and limitations

- In the Week 1 cleaned file, the mean and median are close for each item.
- The first-to-last percentage change compares only the first and last dates.
- The Week 2 flagged-value comparison shows the effect of 1,250 on the mean and median; it does not prove the value is an error.
- Missing values are not filled with invented numbers.
- The data is labelled illustrative teaching data, not actual PBS observations.
